In [ ]:
"""
BigAlpha 2026 —— 34 特征提交版（训练表写死）

与已通过公榜 0.95169 的 25 特征版相比，只改两处：
  FEATURES        25 -> 34（加回当初被判的 A32/A64/A92/A108/A138/A140/A154/A169/A179）
  colsample_bytree 0.68 -> 0.5（34 特征下调出来的值，34x0.5 约 17 列）

动机：那 9 个因子当初被判时，探针的 build_feature_panel 同样用 datasources 取表名，
      可能被「训练表用注入表」这同一个 bug 牵连。若本版通过，说明它们是冤枉的。
      本地同口径 34 特征 ModelScore 4.7978，25 特征 2.6988，差 78%。

训练表写死 TRAIN_BAR1M，预测路径仍严格使用 datasources —— 与已通过版本一致。
LOOKBACK_DAYS 保持 320：A154 的 MEAN(VOLUME,180)+CORR(18) 约需 198 个交易日。

通过 -> 拿回全部特征，B 项应显著上升
被判 -> 那 9 个确实有问题，25 特征即上限
"""

import gc
import time
import numpy as np
import pandas as pd

# ==================================================================
# 零、配置
# ==================================================================
FEATURES = [
    # 自制 7
    "own_f6", "own_f42", "own_mp", "own_f4", "own_rev", "own_vol", "own_acc",
    # 第一轮 16
    "A1", "A7", "A15", "A32", "A56", "A62", "A74", "A80", "A83",
    "A105", "A108", "A148", "A154", "A168", "A179", "A185",
    # 第二轮 11
    "A5", "A16", "A38", "A64", "A92", "A111", "A138", "A140", "A156", "A169", "A176",
]
assert len(FEATURES) == 34 and len(set(FEATURES)) == 34

UNIVERSE_TABLE = 'bigalpha_2026_instruments'   # 成分股表，赛制规定无需替换

# 训练用的分钟表写死，与官方模版第 133 行一致：
#   模版训练段 build_features('bigalpha_2026_financial', 'bigalpha_2026_stock_bar1m', ...)
#   模版预测段 build_features(financial_table, datasources['bar1m'], ...)
# 原因：datasources['bar1m'] 在评测时指向验证集表，按官方速查表它只含
#       评估区间 + 向前约一个季度的缓冲，不含 2020-2023 的训练数据。
# 预测路径仍然严格使用 datasources，不硬编码。
TRAIN_BAR1M = 'bigalpha_2026_stock_bar1m'

# 训练区间，写死。落在公榜训练集范围 2019-01-01 ~ 2024-12-31 内。
# 写法与模版一致：带时分秒，末日补 23:59:59，确保末日整天都在区间内。
TRAIN_START = '2020-01-01 00:00:00'
TRAIN_END   = '2024-12-31 23:59:59'

LOOKBACK_DAYS = 320            # 回到 U1 通过时的取值；本份唯一的变量
PURGE_DAYS = 2                 # 标签 close_{t+1}/close_t 会让 train 末日用到 valid 首日收盘价

BLEND_W = 0.25                 # factor = (1−w)·z(demean预测) + w·z(rank预测)

N_JOBS = 4                     # 实测最优：比 64 快 49.7 倍（cgroup 配额 4 核，开 64 严重超订）

PARAMS = {
    'demean': dict(max_depth=8, min_child_weight=10, colsample_bytree=0.5,
                   subsample=0.6, learning_rate=0.05, reg_lambda=5.0,
                   base_score=0.0, tree_method='hist',
                   objective='reg:squarederror', random_state=42),
    'rank':   dict(max_depth=8, min_child_weight=10, colsample_bytree=0.5,
                   subsample=1.0, learning_rate=0.05, reg_lambda=5.0,
                   base_score=0.0, tree_method='hist',
                   objective='reg:squarederror', random_state=42),
}
N_ESTIMATORS = {'demean': 750, 'rank': 750}

class Timer:
    """分段计时，只做打印，用于确认 main() 在 3 小时限时内"""

    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.time()
        print(f"[计时] 开始: {self.name}", flush=True)
        return self

    def __exit__(self, *a):
        dt = time.time() - self.t0
        print(f"[计时] 完成: {self.name}  {dt/60:.2f} 分钟 ({dt:.1f}s)", flush=True)


# ==================================================================
# 一、算子库（照抄 corr40_dedup.py）
# ==================================================================
def _g(df, col):
    return df.groupby('instrument', sort=False)[col]


def _mp(n, floor=3):
    if n <= 1:
        return 1
    return max(2, min(n, max(floor, n // 2)))


def DELAY(df, col, n):
    return _g(df, col).shift(n)


def DELTA(df, col, n):
    return df[col] - _g(df, col).shift(n)


def SUM(df, col, n):
    return _g(df, col).rolling(n, min_periods=_mp(n, 2)).sum().reset_index(level=0, drop=True)


def MEAN(df, col, n):
    return _g(df, col).rolling(n, min_periods=_mp(n, 2)).mean().reset_index(level=0, drop=True)


def STD(df, col, n):
    return _g(df, col).rolling(n, min_periods=_mp(n, 2)).std(ddof=1).reset_index(level=0, drop=True)


def TSMAX(df, col, n):
    return _g(df, col).rolling(n, min_periods=_mp(n, 2)).max().reset_index(level=0, drop=True)


def TSMIN(df, col, n):
    return _g(df, col).rolling(n, min_periods=_mp(n, 2)).min().reset_index(level=0, drop=True)


def _rolling_windows(a, n):
    from numpy.lib.stride_tricks import sliding_window_view
    return sliding_window_view(np.concatenate([np.full(n - 1, np.nan), a]), n)


def _apply_vec_by_inst(df, col, n, func):
    out = np.full(len(df), np.nan)
    vals = df[col].to_numpy(dtype='float64')
    codes = df['instrument'].to_numpy()
    change = np.flatnonzero(codes[1:] != codes[:-1]) + 1
    starts = np.concatenate([[0], change])
    ends = np.concatenate([change, [len(df)]])
    with np.errstate(invalid='ignore', divide='ignore'):
        for s, e in zip(starts, ends):
            if e - s < 2:
                continue
            out[s:e] = func(_rolling_windows(vals[s:e], n))
    return pd.Series(out, index=df.index)


def TSRANK(df, col, n):
    def f(W):
        last = W[:, -1]
        cnt = (~np.isnan(W)).sum(axis=1)
        le = np.nansum(W <= last[:, None], axis=1)
        r = np.where(cnt >= 2, le / np.maximum(cnt, 1), np.nan)
        return np.where(np.isnan(last), np.nan, r)
    return _apply_vec_by_inst(df, col, n, f)


def CORR(df, c1, c2, n):
    mp = _mp(n)
    out = df.groupby('instrument', sort=False, group_keys=False).apply(
        lambda g: g[c1].rolling(n, min_periods=mp).corr(g[c2]))
    return out.reindex(df.index)


def COV(df, c1, c2, n):
    mp = _mp(n)
    out = df.groupby('instrument', sort=False, group_keys=False).apply(
        lambda g: g[c1].rolling(n, min_periods=mp).cov(g[c2]))
    return out.reindex(df.index)


def RANK(df, s):
    return s.groupby(df['date']).rank(pct=True)


def SIGN(s):
    return np.sign(s)


def ABS(s):
    return s.abs()


def LOG(s):
    return np.log(s.where(s > 0))


def MAXe(a, b):
    if np.isscalar(b):
        return a.clip(lower=b)
    return pd.concat([a, b], axis=1).max(axis=1)


def MINe(a, b):
    if np.isscalar(b):
        return a.clip(upper=b)
    return pd.concat([a, b], axis=1).min(axis=1)


def SMA(df, col, n, m):
    """官方: Y_i=(A_i*m+Y_{i-1}*(n-m))/n；等价 ewm(alpha=m/n, adjust=False)"""
    return _g(df, col).transform(lambda s: s.ewm(alpha=m / n, adjust=False, ignore_na=False).mean())


# ---------- [改动1] DECAYLINEAR 向量化 ----------
# 原实现用 groupby.rolling.apply，34 个因子里共 10 次调用，是特征构造的主要瓶颈。
# 向量化后与原版严格等价，理由：
#   原版对长度 L 的窗口取权重 w_new_to_old[:L][::-1] = [d-L+1, ..., d]，
#   恰好等于满窗权重 [1..d] 的后 L 项；向量化版前补 NaN 凑满窗，
#   NaN 位置权重被 mask 成 0，剩下的实值拿到的正是 [1..d] 的后 L 项。完全一致。
# 与 rolling.apply 原版的数值对拍已在本地验证文件中完成（8 个窗口长度全部等价）。
def DECAYLINEAR(df, col, d):
    w = np.arange(1, d + 1, dtype='float64')   # 由旧到新，最新权重最大

    def f(W):
        mask = ~np.isnan(W)
        cnt = mask.sum(axis=1)
        ww = np.where(mask, w, 0.0)
        s = ww.sum(axis=1)
        num = np.nansum(np.where(mask, W, 0.0) * ww, axis=1)
        ok = (cnt >= 2) & (s > 0)
        return np.where(ok, num / np.where(s == 0, 1.0, s), np.nan)

    return _apply_vec_by_inst(df, col, d, f)


def COND(cond, a, b):
    """A ? B : C 三元；cond 为布尔 Series"""
    return pd.Series(np.where(cond.fillna(False), a, b), index=cond.index)


# ==================================================================
# 二、日频面板
# ==================================================================
def build_panel(datasources, q_start, q_end):
    """bar1m 聚合日频：复权 OHLC/volume + amount + vwap(原始) + ret

    q_start / q_end 已含 LOOKBACK 缓冲，由调用方算好
    """
    import dai

    bar1m = datasources['bar1m']
    df = dai.query(f"""
        SELECT CAST(date AS DATE) AS date, instrument,
               arg_min(open, date)  AS open_raw,
               MAX(high)            AS high_raw,
               MIN(low)             AS low_raw,
               arg_max(close, date) AS close_raw,
               SUM(volume)          AS volume_raw,
               SUM(amount)          AS amount,
               MAX(adjust_factor)   AS adj
        FROM {bar1m} GROUP BY 1, 2
    """, filters={'date': [q_start, q_end]}).df()

    df['date'] = pd.to_datetime(df['date'])
    for c in ['open_raw', 'high_raw', 'low_raw', 'close_raw', 'volume_raw', 'amount', 'adj']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df = df[(df['volume_raw'] > 0) & (df['adj'] > 0)].copy()
    df = df.sort_values(['instrument', 'date']).reset_index(drop=True)

    df['open']   = df['open_raw']  * df['adj']
    df['high']   = df['high_raw']  * df['adj']
    df['low']    = df['low_raw']   * df['adj']
    df['close']  = df['close_raw'] * df['adj']
    df['volume'] = df['volume_raw'] / df['adj']
    df['vwap']   = df['amount'] / df['volume_raw']
    df['ret']    = df['close'] / _g(df, 'close').shift(1) - 1.0
    return df


# ==================================================================
# 三、因子计算
# ==================================================================
def compute_own7(datasources, d, q_start, q_end):
    import dai

    out = {}
    d = d.copy()

    out['own_f6'] = -1.0 * CORR(d, 'open', 'volume', 10)

    d['_close_raw'] = d['close'] / d['adj']
    out['own_f42'] = RANK(d, d['vwap'] - d['_close_raw']) / RANK(d, d['vwap'] + d['_close_raw'])

    d['_r_low'] = RANK(d, d['low'])
    out['own_f4'] = -1.0 * TSRANK(d, '_r_low', 9)

    out['own_rev'] = 1.0 - RANK(d, DELTA(d, 'close', 1))

    d['_std2'] = STD(d, 'ret', 2)
    d['_std5'] = STD(d, 'ret', 5)
    d['_vr'] = d['_std2'] / d['_std5'].replace(0, np.nan)
    out['own_vol'] = 1.0 - RANK(d, d['_vr'])

    _d10 = DELAY(d, 'close', 10)
    _d20 = DELAY(d, 'close', 20)
    out['own_acc'] = -1.0 * ((_d20 - _d10) / 10.0 - (_d10 - d['close']) / 10.0)

    bar1m = datasources['bar1m']
    mp = dai.query(f"""
        WITH pm AS (SELECT CAST(date AS DATE) AS date, instrument,
            (bid_price1+ask_price1)/2.0 AS mid,
            CASE WHEN bid_volume1=0 AND ask_volume1=0 THEN (bid_price1+ask_price1)/2.0
                 WHEN bid_volume1=0 THEN ask_price1 WHEN ask_volume1=0 THEN bid_price1
                 ELSE (bid_price1*ask_volume1+ask_price1*bid_volume1)/(ask_volume1+bid_volume1) END AS wpr
            FROM {bar1m} WHERE bid_price1>0 AND ask_price1>0)
        SELECT date, instrument, AVG((mid-wpr)/mid) AS own_mp FROM pm WHERE mid>0 GROUP BY 1,2
    """, filters={'date': [q_start, q_end]}).df()
    mp['date'] = pd.to_datetime(mp['date'])
    mp['own_mp'] = pd.to_numeric(mp['own_mp'], errors='coerce')

    return out, mp


def compute_r1_16(d):
    """第一轮入选 16 个（原 17 个候选中 A36 被 A16 覆盖已剔除）"""
    f = {}
    d = d.copy()
    d['_hl2'] = (d['high'] + d['low']) / 2
    d['_rv'] = RANK(d, d['volume'])
    d['_rw'] = RANK(d, d['vwap'])
    d['_rh'] = RANK(d, d['high'])
    d['_ro'] = RANK(d, d['open'])
    d['_rl'] = RANK(d, d['low'])

    # A1: -CORR(RANK(DELTA(LOG(VOLUME),1)), RANK((CLOSE-OPEN)/OPEN), 6)
    d['_a'] = RANK(d, DELTA(d.assign(_lv=LOG(d['volume'])), '_lv', 1))
    d['_b'] = RANK(d, (d['close'] - d['open']) / d['open'])
    f['A1'] = -1 * CORR(d, '_a', '_b', 6)

    # A7: (RANK(MAX(VWAP-CLOSE,3)) + RANK(MIN(VWAP-CLOSE,3))) * RANK(DELTA(VOLUME,3))
    f['A7'] = (RANK(d, MAXe(d['vwap'] - d['close'], 3)) + RANK(d, MINe(d['vwap'] - d['close'], 3))) \
              * RANK(d, DELTA(d, 'volume', 3))

    # A15: OPEN/DELAY(CLOSE,1)-1   （隔夜跳空，库里的最大盲区）
    f['A15'] = d['open'] / DELAY(d, 'close', 1) - 1

    # A32: -SUM(RANK(CORR(RANK(HIGH),RANK(VOLUME),3)),3)
    d['_c32'] = RANK(d, CORR(d, '_rh', '_rv', 3))
    f['A32'] = -1 * SUM(d, '_c32', 3)

    # [改动2] A36 已剔除（被 A16 覆盖，corr −0.711）；其独占中间量 _c36 一并删除

    # A56: (RANK(OPEN-TSMIN(OPEN,12)) < RANK(RANK(CORR(SUM((HIGH+LOW)/2,19),SUM(MEAN(VOLUME,40),19),13))^5))
    d['_s_hl2_19'] = SUM(d, '_hl2', 19)
    d['_m40'] = MEAN(d, 'volume', 40)
    d['_s_m40_19'] = SUM(d, '_m40', 19)
    d['_c56'] = RANK(d, CORR(d, '_s_hl2_19', '_s_m40_19', 13))
    f['A56'] = (RANK(d, d['open'] - TSMIN(d, 'open', 12)) < RANK(d, d['_c56'] ** 5)).astype(float)

    # A62: -CORR(HIGH, RANK(VOLUME), 5)
    f['A62'] = -1 * CORR(d, 'high', '_rv', 5)

    # A74: RANK(CORR(SUM(LOW*0.35+VWAP*0.65,20),SUM(MEAN(VOLUME,40),20),7)) + RANK(CORR(RANK(VWAP),RANK(VOLUME),6))
    d['_lw'] = d['low'] * 0.35 + d['vwap'] * 0.65
    d['_s_lw'] = SUM(d, '_lw', 20)
    d['_s_m40_20'] = SUM(d, '_m40', 20)
    f['A74'] = RANK(d, CORR(d, '_s_lw', '_s_m40_20', 7)) + RANK(d, CORR(d, '_rw', '_rv', 6))

    # A80: (VOLUME-DELAY(VOLUME,5))/DELAY(VOLUME,5)*100
    _dv5 = DELAY(d, 'volume', 5)
    f['A80'] = (d['volume'] - _dv5) / _dv5 * 100

    # A83: -RANK(COV(RANK(HIGH),RANK(VOLUME),5))
    f['A83'] = -1 * RANK(d, COV(d, '_rh', '_rv', 5))

    # A105: -CORR(RANK(OPEN),RANK(VOLUME),10)
    f['A105'] = -1 * CORR(d, '_ro', '_rv', 10)

    # A108: (RANK(HIGH-TSMIN(HIGH,2)) ^ RANK(CORR(VWAP,MEAN(VOLUME,120),6))) * -1
    d['_m120'] = MEAN(d, 'volume', 120)
    d['_c108'] = RANK(d, CORR(d, 'vwap', '_m120', 6))
    f['A108'] = (RANK(d, d['high'] - TSMIN(d, 'high', 2)) ** d['_c108']) * -1

    # A148: (RANK(CORR(OPEN,SUM(MEAN(VOLUME,60),9),6)) < RANK(OPEN-TSMIN(OPEN,14))) * -1
    d['_m60'] = MEAN(d, 'volume', 60)
    d['_s_m60_9'] = SUM(d, '_m60', 9)
    _l148 = RANK(d, CORR(d, 'open', '_s_m60_9', 6))
    _r148 = RANK(d, d['open'] - TSMIN(d, 'open', 14))
    f['A148'] = ((_l148 < _r148).astype(float)) * -1

    # A154: (VWAP-TSMIN(VWAP,16)) < CORR(VWAP,MEAN(VOLUME,180),18)
    d['_m180'] = MEAN(d, 'volume', 180)
    f['A154'] = ((d['vwap'] - TSMIN(d, 'vwap', 16)) < CORR(d, 'vwap', '_m180', 18)).astype(float)

    # A168: -VOLUME/MEAN(VOLUME,20)
    f['A168'] = -1 * d['volume'] / MEAN(d, 'volume', 20)

    # A179: RANK(CORR(VWAP,VOLUME,4)) * RANK(CORR(RANK(LOW),RANK(MEAN(VOLUME,50)),12))
    d['_rm50'] = RANK(d, MEAN(d, 'volume', 50))
    f['A179'] = RANK(d, CORR(d, 'vwap', 'volume', 4)) * RANK(d, CORR(d, '_rl', '_rm50', 12))

    # A185: RANK(-(1-OPEN/CLOSE)^2)
    f['A185'] = RANK(d, -1 * ((1 - d['open'] / d['close']) ** 2))

    return f


def compute_r2_11(d):
    """第二轮入选 11 个（原 16 个候选中 A86/A102/A109/A152/A155 已剔除）"""
    f = {}
    d = d.copy()
    d['_rv'] = RANK(d, d['volume'])
    d['_rw'] = RANK(d, d['vwap'])
    d['_ro'] = RANK(d, d['open'])
    d['_rc'] = RANK(d, d['close'])
    d['_rl'] = RANK(d, d['low'])
    d['_rh'] = RANK(d, d['high'])
    _dc1 = DELAY(d, 'close', 1)          # A169 仍需要（A86 已剔除）

    # A5: -TSMAX(CORR(TSRANK(VOLUME,5), TSRANK(HIGH,5), 5), 3)
    d['_tv5'] = TSRANK(d, 'volume', 5)
    d['_th5'] = TSRANK(d, 'high', 5)
    d['_c5'] = CORR(d, '_tv5', '_th5', 5)
    f['A5'] = -1 * TSMAX(d, '_c5', 3)

    # A16: -TSMAX(RANK(CORR(RANK(VOLUME),RANK(VWAP),5)), 5)
    d['_c16'] = RANK(d, CORR(d, '_rv', '_rw', 5))
    f['A16'] = -1 * TSMAX(d, '_c16', 5)

    # A38: (SUM(HIGH,20)/20 < HIGH) ? (-DELTA(HIGH,2)) : 0
    f['A38'] = COND(SUM(d, 'high', 20) / 20 < d['high'], -1 * DELTA(d, 'high', 2), 0.0)

    # A64: (MAX(RANK(DECAYLINEAR(CORR(RANK(VWAP),RANK(VOLUME),4),4)),
    #           RANK(DECAYLINEAR(MAX(CORR(RANK(CLOSE),RANK(MEAN(VOLUME,60)),4),13),14))) * -1)
    d['_c64a'] = CORR(d, '_rw', '_rv', 4)
    d['_a64a'] = RANK(d, DECAYLINEAR(d, '_c64a', 4))
    d['_rm60'] = RANK(d, MEAN(d, 'volume', 60))
    d['_c64b'] = CORR(d, '_rc', '_rm60', 4)
    d['_mx64'] = TSMAX(d, '_c64b', 13)
    d['_a64b'] = RANK(d, DECAYLINEAR(d, '_mx64', 14))
    f['A64'] = MAXe(d['_a64a'], d['_a64b']) * -1

    # [改动2] A86 已剔除（被 own_acc 覆盖，corr +0.759）

    # A92: (MAX(RANK(DECAYLINEAR(DELTA(CLOSE*0.35+VWAP*0.65,2),3)),
    #           TSRANK(DECAYLINEAR(ABS(CORR(MEAN(VOLUME,180),CLOSE,13)),5),15)) * -1)
    d['_mix92'] = d['close'] * 0.35 + d['vwap'] * 0.65
    d['_d92'] = DELTA(d, '_mix92', 2)
    d['_m180'] = MEAN(d, 'volume', 180)
    d['_c92'] = ABS(CORR(d, '_m180', 'close', 13))
    d['_dl92'] = DECAYLINEAR(d, '_c92', 5)
    f['A92'] = MAXe(RANK(d, DECAYLINEAR(d, '_d92', 3)), TSRANK(d, '_dl92', 15)) * -1

    # [改动2] A102 已剔除（被 A168 覆盖，corr −0.906）
    # [改动2] A109 已剔除（被 A168 覆盖，corr −0.704）；但其中间量 _hl 被 A111 共用，保留

    d['_hl'] = d['high'] - d['low']

    # A111: SMA(VOLUME*((CLOSE-LOW)-(HIGH-CLOSE))/(HIGH-LOW),11,2) - SMA(同,4,2)
    d['_x111'] = d['volume'] * ((d['close'] - d['low']) - (d['high'] - d['close'])) / d['_hl']
    f['A111'] = SMA(d, '_x111', 11, 2) - SMA(d, '_x111', 4, 2)

    # A138: ((RANK(DECAYLINEAR(DELTA(LOW*0.7+VWAP*0.3,3),20))
    #         - TSRANK(DECAYLINEAR(TSRANK(CORR(TSRANK(LOW,8),TSRANK(MEAN(VOLUME,60),17),5),19),16),7)) * -1)
    d['_mix138'] = d['low'] * 0.7 + d['vwap'] * 0.3
    d['_d138'] = DELTA(d, '_mix138', 3)
    d['_tl8'] = TSRANK(d, 'low', 8)
    d['_m60'] = MEAN(d, 'volume', 60)
    d['_tm60'] = TSRANK(d, '_m60', 17)
    d['_c138'] = CORR(d, '_tl8', '_tm60', 5)
    d['_t138'] = TSRANK(d, '_c138', 19)
    d['_dl138'] = DECAYLINEAR(d, '_t138', 16)
    f['A138'] = (RANK(d, DECAYLINEAR(d, '_d138', 20)) - TSRANK(d, '_dl138', 7)) * -1

    # A140: MIN(RANK(DECAYLINEAR((RANK(OPEN)+RANK(LOW))-(RANK(HIGH)+RANK(CLOSE)),8)),
    #           TSRANK(DECAYLINEAR(CORR(TSRANK(CLOSE,8),TSRANK(MEAN(VOLUME,60),20),8),7),3))
    d['_x140'] = (d['_ro'] + d['_rl']) - (d['_rh'] + d['_rc'])
    d['_tc8'] = TSRANK(d, 'close', 8)
    d['_tm60b'] = TSRANK(d, '_m60', 20)
    d['_c140'] = CORR(d, '_tc8', '_tm60b', 8)
    d['_dl140'] = DECAYLINEAR(d, '_c140', 7)
    f['A140'] = MINe(RANK(d, DECAYLINEAR(d, '_x140', 8)), TSRANK(d, '_dl140', 3))

    # [改动2] A152 已剔除（被 A169 覆盖，corr +0.702）
    # [改动2] A155 已剔除（被 A168 覆盖，corr −0.716）

    # A156: (MAX(RANK(DECAYLINEAR(DELTA(VWAP,5),3)),
    #            RANK(DECAYLINEAR((DELTA(OPEN*0.15+LOW*0.85,2)/(OPEN*0.15+LOW*0.85))*-1,3))) * -1)
    d['_dvw5'] = DELTA(d, 'vwap', 5)
    d['_mix156'] = d['open'] * 0.15 + d['low'] * 0.85
    d['_x156'] = (DELTA(d, '_mix156', 2) / d['_mix156']) * -1
    f['A156'] = MAXe(RANK(d, DECAYLINEAR(d, '_dvw5', 3)),
                     RANK(d, DECAYLINEAR(d, '_x156', 3))) * -1

    # A169: SMA(MEAN(DELAY(SMA(CLOSE-DELAY(CLOSE,1),9,1),1),12) - MEAN(同,26), 10, 1)
    d['_d1'] = d['close'] - _dc1
    d['_s169'] = SMA(d, '_d1', 9, 1)
    d['_ds169'] = DELAY(d, '_s169', 1)
    d['_x169'] = MEAN(d, '_ds169', 12) - MEAN(d, '_ds169', 26)
    f['A169'] = SMA(d, '_x169', 10, 1)

    # A176: CORR(RANK((CLOSE-TSMIN(LOW,12))/(TSMAX(HIGH,12)-TSMIN(LOW,12))), RANK(VOLUME), 6)
    _tl12 = TSMIN(d, 'low', 12)
    _th12 = TSMAX(d, 'high', 12)
    d['_k176'] = RANK(d, (d['close'] - _tl12) / (_th12 - _tl12))
    f['A176'] = CORR(d, '_k176', '_rv', 6)

    return f


# ==================================================================
# 四、特征面板组装
# ==================================================================
def build_feature_panel(datasources, panel_start, panel_end, universe_start, universe_end,
                        with_label=True, verbose=True):
    """返回对齐股票池后的宽表：date, instrument, 34 特征 [, y]"""
    import dai

    # 传入的可能是平台注入的原始字符串，带时分秒（如 '2025-01-01 00:00:00'）。
    # 面板是日频，所有边界一律归到整日，避免带时刻的下界把首日整天筛掉。
    ps = pd.to_datetime(panel_start).normalize()
    q0 = (ps - pd.Timedelta(days=LOOKBACK_DAYS)).strftime('%Y-%m-%d 00:00:00')
    q1 = pd.to_datetime(panel_end).normalize().strftime('%Y-%m-%d 23:59:59')
    u0 = pd.to_datetime(universe_start).normalize().strftime('%Y-%m-%d 00:00:00')
    u1 = pd.to_datetime(universe_end).normalize().strftime('%Y-%m-%d 23:59:59')

    with Timer("build_panel (bar1m 日频聚合)"):
        d = build_panel(datasources, q0, q1)
        if verbose:
            print(f"       原始面板 {len(d):,} 行, {d['instrument'].nunique()} 只股票, "
                  f"{d['date'].min().date()} ~ {d['date'].max().date()}")

    with Timer("自制 7 因子 (含 own_mp 盘口查询)"):
        own, mp = compute_own7(datasources, d, q0, q1)

    with Timer("第一轮 16 个 A 因子"):
        r1 = compute_r1_16(d)

    with Timer("第二轮 11 个 A 因子"):
        r2 = compute_r2_11(d)

    with Timer("拼宽表 + 对齐股票池"):
        panel = d[['date', 'instrument']].copy()
        if with_label:
            # 标签：close_{t+1}/close_t - 1，与平台评测口径一致
            # 注意：shift(-1) 是按行位移，停牌导致的数据缺行会让"下一行"不是下一交易日，
            #       这是本地口径的已知近似，平台按真实交易日算
            panel['y'] = _g(d, 'close').shift(-1) / d['close'] - 1.0

        for src in (own, r1, r2):
            for k in src:
                if k == 'own_mp':          # own_mp 走 merge，不在 d.index 上
                    continue
                panel[k] = pd.to_numeric(
                    pd.Series(np.asarray(src[k], dtype='float64'), index=d.index), errors='coerce')

        panel = pd.merge(panel, mp, how='left', on=['date', 'instrument'])

        del d, own, r1, r2, mp
        gc.collect()

        missing = [c for c in FEATURES if c not in panel.columns]
        assert not missing, f"缺失特征列: {missing}"

        panel[FEATURES] = panel[FEATURES].replace([np.inf, -np.inf], np.nan)
        panel = panel[panel['date'] >= ps]

        stk = dai.query(f"SELECT date, instrument FROM {UNIVERSE_TABLE}",
                        filters={'date': [u0, u1]}).df()
        stk['date'] = pd.to_datetime(stk['date'])
        panel = pd.merge(panel, stk, how='inner', on=['date', 'instrument'])
        panel = panel.sort_values(['date', 'instrument']).reset_index(drop=True)

    if verbose:
        cov = panel[FEATURES].notna().mean().sort_values()
        print(f"\n特征面板: {len(panel):,} 行 × {len(FEATURES)} 特征, "
              f"{panel['date'].min().date()} ~ {panel['date'].max().date()}")
        print("覆盖度最低的 8 个特征:")
        for k, v in cov.head(8).items():
            print(f"    {k:<10s} {v:.4f}")
        print(f"覆盖度中位数 {cov.median():.4f}, 最高 {cov.max():.4f}")

    return panel



# ==================================================================
# 五、标签口径与归一化
# ==================================================================
def make_label(dates, y, mode):
    """
    次日收益 y(i,t) = m(t) + e(i,t)
      m(t)   当天全市场共同涨跌 —— 平台逐日截面排序时被完全抹掉，一分不给
      e(i,t) 个股相对市场的偏离 —— 只有这块算分

    raw 标签下梯度里含 −m(t)（当日所有股票共用的偏移），模型有动机去猜大盘方向，
    而能指示大盘方向的恰是波动率/成交量/市值这类风格特征 —— 风格暴露的来源。
    demean 逐日 z-score：减均值消掉 m(t)，除标准差让各天在损失里权重相等。
    rank   逐日排名减当日均值：只保留次序。
    """
    y = np.asarray(y, dtype='float64')
    d = pd.Series(pd.to_datetime(dates)).values
    s = pd.Series(y)
    g = s.groupby(d)
    if mode == 'demean':
        return ((s - g.transform('mean')) / g.transform('std').replace(0, np.nan)).to_numpy()
    if mode == 'rank':
        # 不能直接减 0.5：rank(pct=True) 是 1/n..n/n，均值 (n+1)/(2n)，
        # 减 0.5 后每天残留 1/(2n_t)，而 n_t 逐日变化
        r = g.rank(pct=True)
        return (r - r.groupby(d).transform('mean')).to_numpy()
    raise ValueError(mode)


def daily_zscore(dates, v):
    """逐日截面 z-score —— 混合前的归一化。

    选它而不选秩变换：这是逐日仿射变换，会被平台的「去极值 + 逐日标准化」
    完全吸收（实测中性化后残差的秩相关 = 1.00000），
    所以 w=0 精确等于纯 demean、w=1 精确等于纯 rank，混合曲线可解释。
    秩变换不是仿射，会改变风格中性化后残差的排序，实测 Sharpe 从 2.50 变 −0.19。
    """
    s = pd.Series(np.asarray(v, dtype='float64'))
    g = s.groupby(pd.Series(pd.to_datetime(dates)).values)
    return ((s - g.transform('mean')) / g.transform('std').replace(0, np.nan)
            ).fillna(0.0).to_numpy()


# ==================================================================
# 六、提交入口
# ==================================================================
def main(datasources, start_date, end_date):
    """
    平台只替换 datasources / start_date / end_date，后两者是【测试集区间】。
    训练区间在此写死为 TRAIN_START ~ TRAIN_END，绝不使用传入的 start/end 训练。

    返回 ['date', 'instrument', 'factor']，无 inf，已对齐中证 1000 成分股。
    """
    import xgboost as xgb

    print(f"训练区间(写死) {TRAIN_START} ~ {TRAIN_END}")
    print(f"测试区间(平台注入) {start_date} ~ {end_date}")

    # --- 训练面板：只用写死区间构造，与传入的 start_date / end_date 无关 ---
    # 训练面板与测试面板分开构造，各带自己的 LOOKBACK 缓冲。
    # 若合成一次查询，当两段之间存在无数据的年份时，滚动窗口会跨着数据空洞计算
    # （MEAN(VOLUME,180) 把两年前的成交量当近期历史），不报错但因子值全错。
    with Timer("构造训练面板"):
        train = build_feature_panel({'bar1m': TRAIN_BAR1M}, TRAIN_START, TRAIN_END,
                                    TRAIN_START, TRAIN_END,
                                    with_label=True, verbose=True)

    # --- 测试面板：只用传入区间构造，不计算标签 ---
    # with_label=False 使得全文唯一的 shift(-1) 只作用于训练面板。
    with Timer("构造测试面板"):
        test = build_feature_panel(datasources, start_date, end_date,
                                   start_date, end_date,
                                   with_label=False, verbose=True)

    print(f"    [训练面板] {len(train):,} 行  "
          f"{train['date'].min() if len(train) else '空'} ~ "
          f"{train['date'].max() if len(train) else '空'}  "
          f"{train['date'].nunique() if len(train) else 0} 个交易日")

    if len(train) == 0 or len(test) == 0:
        print("训练面板或测试面板为空，返回空表")
        return pd.DataFrame(columns=['date', 'instrument', 'factor'])

    # --- purge + 训练/测试隔离 ---
    # purge：标签 close_{t+1}/close_t 会让训练末日用到次日收盘价，
    #        砍掉训练面板末尾 PURGE_DAYS 个交易日。
    # 隔离：训练样本的日期再截断到测试区间起点之前。这一条不依赖任何入参假设，
    #      因此无论平台注入什么区间，训练集与测试集都不可能相交。
    te_lo = test['date'].min()
    days = np.sort(train['date'].unique())
    cut = pd.Timestamp(days[-PURGE_DAYS]) if len(days) > PURGE_DAYS else pd.Timestamp(days[0])
    cut = min(cut, te_lo)

    tr = train[train['date'] < cut]
    tr = tr[np.isfinite(tr['y'])].copy()
    del train
    gc.collect()

    if len(tr) == 0:
        raise RuntimeError(
            f"训练集为空：测试区间起点 {te_lo.date()} 不晚于训练区间起点 {TRAIN_START}")
    assert tr['date'].max() < te_lo, "训练区间与测试区间相交"

    te = test
    print(f"训练 {len(tr):,} 行 ({tr['date'].min().date()} ~ {tr['date'].max().date()})   "
          f"预测 {len(te):,} 行 ({te['date'].min().date()} ~ {te['date'].max().date()})")

    # --- 两个模型，各自的超参与轮数 ---
    Xte = te[FEATURES]
    z = {}
    for lab in ('demean', 'rank'):
        with Timer(f"训练 {lab} 模型 ({N_ESTIMATORS[lab]} 轮)"):
            ytr = make_label(tr['date'], tr['y'].to_numpy(), lab)
            ok = np.isfinite(ytr)          # z-score 遇到当日 std=0 会产生 NaN
            mdl = xgb.XGBRegressor(**PARAMS[lab], n_estimators=N_ESTIMATORS[lab],
                                   n_jobs=N_JOBS)
            mdl.fit(tr.loc[tr.index[ok], FEATURES], ytr[ok], verbose=False)
            z[lab] = daily_zscore(te['date'], mdl.predict(Xte))
            del mdl
            gc.collect()

    # --- 混合 ---
    te['factor'] = (1 - BLEND_W) * z['demean'] + BLEND_W * z['rank']

    out = te[['date', 'instrument', 'factor']].copy()
    out['instrument'] = out['instrument'].astype(str)
    out['factor'] = pd.to_numeric(out['factor'], errors='coerce')
    out = out.replace([np.inf, -np.inf], np.nan).dropna(subset=['factor'])
    out = out.sort_values(['date', 'instrument']).reset_index(drop=True)

    cov = out.groupby('date').size()
    print(f"输出 {len(out):,} 行, {out['date'].nunique()} 个交易日, "
          f"每日股票数 中位 {cov.median():.0f} 最少 {cov.min():.0f}")
    return out